In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

import regex
import evaluate
import torch

c:\Users\User\.conda\envs\torch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Loading & Inspecting data

In [2]:
train_data = pd.read_csv("dataset/samsum-train.csv")
val_data = pd.read_csv("dataset/samsum-validation.csv")
test_data = pd.read_csv("dataset/samsum-train.csv")

In [14]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [15]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 14732 entries, 0 to 14731
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        14732 non-null  str  
 1   dialogue  14731 non-null  str  
 2   summary   14732 non-null  str  
dtypes: str(3)
memory usage: 9.2 MB


The number of datapoints are too much to train on local system. Only using sample of it.

In [ ]:
train_data = train_data.sample(2000, random_state=42).reset_index()
val_data = val_data.sample(400, random_state=42).reset_index()
test_data = test_data.sample(200, random_state=42).reset_index()

# Pre-processing

### Remove unnecessary columns

In [19]:
train_data.columns

Index(['index', 'id', 'dialogue', 'summary'], dtype='str')

In [ ]:
train_data = train_data[["dialogue", "summary"]]
val_data = val_data[["dialogue", "summary"]]

### Missing values

In [21]:
train_data.isnull().sum()

dialogue    0
summary     0
dtype: int64

In [22]:
val_data.isnull().sum()

dialogue    0
summary     0
dtype: int64

### Duplicated values

In [23]:
train_data.duplicated().sum()

np.int64(0)

### Clean data

In [4]:
def clean_data(text):
    text = regex.sub(r"\r\n", " ", text) # escape sequences
    text = regex.sub(r"\s+", " ", text) # reduce more than one spaces to one 
    text = regex.sub(r"<.*?>", " ", text) # remove html tags
    text.strip() # remove starting and ending spaces
    return text

In [5]:
train_data["dialogue"] = train_data["dialogue"].map(clean_data)
train_data["summary"] = train_data["summary"].map(clean_data)
val_data["dialogue"] = val_data["dialogue"].map(clean_data)
val_data["summary"] = val_data["summary"].map(clean_data)

### Tokenize

In [8]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
def tokenize(data):
    data["eng"] = ["translate english to urdu " + element for element in data["eng"]]
    inputs = tokenizer(data["eng"], max_length=256, truncation=True)
    outputs = tokenizer(data["urdu"], max_length=256, truncation=True)
    inputs["labels"] = outputs["input_ids"]
    return inputs

In [10]:
train_dataset = train_data.apply(tokenize, axis=1)
val_dataset = val_data.apply(tokenize, axis=1)

# Transfer learning - Transformer

### Load the pre-trained Model

In [12]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 2187.12it/s]


### Fine Tuning

In [11]:
# Make code general for processor
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
device = torch.device(device)
model.to(device)

NameError: name 'model' is not defined

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    weight_decay=0.01,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500,
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


### Saving and loading the model

In [ ]:
# Saving
model.save("./saved_summary_model")
tokenizer.save("./saved_summary_model")

In [13]:
# Loading
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1077.47it/s]


### Test the model

#### Single output testing

In [37]:
def summarize_text(text):
    text = clean_data(text)
    input_tokens = tokenizer(
        "summarize:" + text, 
        max_length=512, 
        padding="max_length", 
        truncation=True, 
        return_tensors="pt"
    )
    model.to(device)
    output_tokens = model.generate(
        input_ids=input_tokens["input_ids"],
        attention_mask=input_tokens["attention_mask"],
        max_new_tokens=128,
        num_beams=4,
        early_stopping=True
    )
    summary = tokenizer.decode(
        output_tokens[0],
        skip_special_tokens=True,
    )
    return summary

In [47]:
# Dialogue on climate change
test_dialogue = """"Ollie: Hi , are you in Warsaw
Jane: yes, just back! Btw are you free for diner the 19th?
Ollie: nope!
Jane: and the  18th?
Ollie: nope, we have this party and you must be there, remember?
Jane: oh right! i lost my calendar..  thanks for reminding me
Ollie: we have lunch this week?
Jane: with pleasure!
Ollie: friday?
Jane: ok
Jane: what do you mean "" we don't have any more whisky!"" lol..
Ollie: what!!!
Jane: you just call me and the all thing i heard was that sentence about whisky... what's wrong with you?
Ollie: oh oh... very strange! i have to be carefull may be there is some spy in my mobile! lol
Jane: dont' worry, we'll check on friday.
Ollie: don't forget to bring some sun with you
Jane: I can't wait to be in Morocco..
Ollie: enjoy and see you friday
Jane: sorry Ollie, i'm very busy, i won't have time for lunch  tomorrow, but may be at 6pm after my courses?this trip to Morocco was so nice, but time consuming!
Ollie: ok for tea!
Jane: I'm on my way..
Ollie: tea is ready, did you bring the pastries?
Jane: I already ate them all... see you in a minute
Ollie: ok"
"""

In [45]:
summarize_text(test_dialogue)

{'input_ids': tensor([[21603,    10,   121,   667,    40,  1896,    10,  2018,     3,     6,
            33,    25,    16,  1602, 13125,  8158,    10,  4273,     6,   131,
           223,    55,   272,    17,   210,    33,    25,   339,    21,   176,
            49,     8,   957,   189,    58,  5424,  1896,    10,   150,   855,
            55,  8158,    10,    11,     8,   507,   189,    58,  5424,  1896,
            10,   150,   855,     6,    62,    43,    48,  1088,    11,    25,
           398,    36,   132,     6,  1423,    58,  8158,    10,     3,    32,
           107,   269,    55,     3,    23,  1513,    82,  4793,     5,     5,
          2049,    21,  5607,    53,   140,  5424,  1896,    10,    62,    43,
          3074,    48,   471,    58,  8158,    10,    28,  5565,    55,  5424,
          1896,    10,  9030,  1135,    58,  8158,    10,     3,  1825,  8158,
            10,   125,   103,    25,  1243,    96,   121,    62,   278,    31,
            17,    43,   136,    72, 1

'"ollie: yes, just back! we have this party and you must be there, remember? ollie: ok Jane: what do you mean "we don\'t have any more whisky!"'

#### Evaluation

In [38]:
predicted_outputs = []
actual_outputs = []
count = 1

for tuple in test_data.itertuples():
    print(f"Iteration: {count}")
    count += 1

    text = clean_data(tuple.dialogue)
    actual_summary = clean_data(tuple.summary)
    predicted_summary = summarize_text(text)

    predicted_outputs.append(predicted_summary)
    actual_outputs.append(actual_summary)

Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Iteration: 6
Iteration: 7
Iteration: 8
Iteration: 9
Iteration: 10
Iteration: 11
Iteration: 12
Iteration: 13
Iteration: 14
Iteration: 15
Iteration: 16
Iteration: 17
Iteration: 18
Iteration: 19
Iteration: 20
Iteration: 21
Iteration: 22
Iteration: 23
Iteration: 24
Iteration: 25
Iteration: 26
Iteration: 27
Iteration: 28
Iteration: 29
Iteration: 30
Iteration: 31
Iteration: 32
Iteration: 33
Iteration: 34
Iteration: 35
Iteration: 36
Iteration: 37
Iteration: 38
Iteration: 39
Iteration: 40
Iteration: 41
Iteration: 42
Iteration: 43
Iteration: 44
Iteration: 45
Iteration: 46
Iteration: 47
Iteration: 48
Iteration: 49
Iteration: 50
Iteration: 51
Iteration: 52
Iteration: 53
Iteration: 54
Iteration: 55
Iteration: 56
Iteration: 57
Iteration: 58
Iteration: 59
Iteration: 60
Iteration: 61
Iteration: 62
Iteration: 63
Iteration: 64
Iteration: 65
Iteration: 66
Iteration: 67
Iteration: 68
Iteration: 69
Iteration: 70
Iteration: 71
Iteration: 72
I

In [39]:
rouge = evaluate.load("rouge")
rouge_score = rouge.compute(predictions=predicted_outputs, references=actual_outputs)
print(f"Rouge Score: {rouge_score}")

Rouge Score: {'rouge1': np.float64(0.45400476680774093), 'rouge2': np.float64(0.21545083420624492), 'rougeL': np.float64(0.38302305780690815), 'rougeLsum': np.float64(0.38282848132563796)}
